In [1]:
import joblib
import pandas as pd
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

In [ ]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["Charge_Per_Tenure_Ratio"] = df["TotalCharges"] / (df["tenure"] + 1e-5)
    df["Tenure_Monthly_Interaction"] = df["tenure"] * df["MonthlyCharges"]
    return df

In [3]:
def get_preprocessing_pipeline(num_cols: list[str], cat_cols: list[str]) -> ColumnTransformer:
    num_pipe = ImbPipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    cat_pipe = ImbPipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", drop="first")),
    ])
    return ColumnTransformer([("num", num_pipe, num_cols), ("cat", cat_pipe, cat_cols)])

In [ ]:
def run_churn_pipeline(df: pd.DataFrame):
    df_fe = engineer_features(df)
    X = df_fe.drop(columns=["customerID", "Churn"], errors="ignore")
    y = df_fe["Churn"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42, stratify=y
    )

    num_cols = ["tenure", "MonthlyCharges", "TotalCharges", "Charge_Per_Tenure_Ratio", "Tenure_Monthly_Interaction"]
    cat_cols = ["Contract", "InternetService", "PaymentMethod", "PaperlessBilling", "SeniorCitizen"]
    preprocessor = get_preprocessing_pipeline(num_cols, cat_cols)

    models = {
        "LogisticRegression": (
            LogisticRegression(max_iter=1000, random_state=42),
            {"classifier__C": [0.01, 0.1, 1.0, 10.0]}
        ),
        "RandomForest": (
            RandomForestClassifier(random_state=42),
            {"classifier__n_estimators": [100, 200], "classifier__max_depth": [5, 10, 15]}
        ),
        "XGBoost": (
            XGBClassifier(random_state=42, eval_metric="logloss"),
            {"classifier__n_estimators": [100, 200], "classifier__learning_rate": [0.01, 0.1]}
        ),
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    best_pipe, best_score = None, -1

    for name, (clf, params) in models.items():
        pipe = ImbPipeline([
            ("preprocessor", preprocessor),
            ("smote", SMOTE(random_state=42)),
            ("classifier", clf),
        ])
        search = RandomizedSearchCV(pipe, params, n_iter=4, scoring="roc_auc", cv=cv, random_state=42, n_jobs=-1)
        search.fit(X_train, y_train)

        score = search.best_score_
        print(f"{name} Best CV ROC-AUC: {score:.4f}")

        if score > best_score:
            best_score = score
            best_pipe = search.best_estimator_

    # Evaluation & Serialization
    preds = best_pipe.predict(X_test)
    probs = best_pipe.predict_proba(X_test)[:, 1]
    
    print(f"\nWinning Model Test ROC-AUC: {roc_auc_score(y_test, probs):.4f}")
    print(classification_report(y_test, preds))

    joblib.dump(best_pipe, "churn_champion_pipeline.joblib")